In [15]:
# ============================================================
# 1. IMPORTS
# ============================================================

import sys
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

print("Imports loaded successfully")

Imports loaded successfully


In [16]:
# ============================================================
# 2. PROJECT PATH AND INTERNAL MODULES
# ============================================================

project_root = Path(r"C:\Users\HUGO\Desktop\Q8 - NORUEGA\TFG\tfg")

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import src.data_pipeline
import src.events

importlib.reload(src.data_pipeline)
importlib.reload(src.events)

from src.data_pipeline import ModelDatasetBuilder
from src.events import SimpleEventDetector, CombinedEventDetector

print("Internal modules loaded successfully")

Internal modules loaded successfully


In [17]:
# ============================================================
# 3. DATABASE PATH AND BUILDER
# ============================================================

db_path = project_root / "NordPoool" / "data" / "thesis_database.db"

print("DB exists:", db_path.exists())
print("DB path:", db_path)

builder = ModelDatasetBuilder(db_path)

print("Builder created successfully")

DB exists: True
DB path: C:\Users\HUGO\Desktop\Q8 - NORUEGA\TFG\tfg\NordPoool\data\thesis_database.db
Builder created successfully


In [19]:
# ============================================================
# 4. LOAD DATA FOR LSTM - NORWAY ZONES
# ============================================================

norway_zones = ["NO1", "NO2", "NO3", "NO4", "NO5"]

zone_dfs = []

for zone in norway_zones:
    print(f"Loading zone: {zone}")

    df_zone = builder.build_price_dataset(
        zones=zone,
        start_date="2020-01-01",
        end_date="2020-12-31",
        add_time_features=True,
        lags=[1, 2, 24, 48, 168],
        target_horizon=1,
        include_volumes=True,
        dropna=False
    )

    # Make sure zone_id is correct
    df_zone["zone_id"] = zone

    zone_dfs.append(df_zone)

df = pd.concat(zone_dfs, axis=0)

df = df.sort_values(["zone_id", "datetime"] if "datetime" in df.columns else ["zone_id"])

print("Dataset shape:", df.shape)
print("Zones:", df["zone_id"].unique())
print("Start:", df.index.min())
print("End:", df.index.max())

df.head()

Loading zone: NO1
Loading zone: NO2
Loading zone: NO3
Loading zone: NO4
Loading zone: NO5
Dataset shape: (43805, 17)
Zones: <StringArray>
['NO1', 'NO2', 'NO3', 'NO4', 'NO5']
Length: 5, dtype: str
Start: 2020-01-01 00:00:00
End: 2020-12-31 00:00:00


,price_id,zone_id,delivery_day,hour,price_value,buy_volume_value,sell_volume_value,year,month,day,day_of_week,price_value_lag_1,price_value_lag_2,price_value_lag_24,price_value_lag_48,price_value_lag_168,target
datetime,,,,,,,,,,,,,,,,,
2020-01-01 00:00:00,32,NO1,2020-01-01,0,31.77,4091.8,1819.6,2020,1,1,2,NaN,NaN,NaN,NaN,NaN,31.57
2020-08-31 17:00:00,117012,NO1,2020-08-31,17,15.28,3103.5,1481.5,2020,8,31,0,15.37,15.16,12.77,10.02,3.91,15.25
2020-08-31 16:00:00,116992,NO1,2020-08-31,16,15.37,3106.5,1482.4,2020,8,31,0,15.16,15.23,12.66,10.08,3.98,15.28
2020-08-31 15:00:00,116972,NO1,2020-08-31,15,15.16,3142.2,1481.8,2020,8,31,0,15.23,15.53,12.57,10.09,4.12,15.37
2020-08-31 14:00:00,116952,NO1,2020-08-31,14,15.23,3186.1,1514.8,2020,8,31,0,15.53,15.57,12.54,10.03,4.18,15.16


In [22]:
# ============================================================
# 5. PRICE, VOLUME AND COMBINED EVENT DETECTION
# ============================================================

detector = SimpleEventDetector()
combined_detector = CombinedEventDetector()

# Price events
df_events = detector.detect_price_events(df)

# Volume events
df_events = detector.detect_volume_events(df_events)

# Combined events
df_events = combined_detector.detect_combined_events(df_events)

print("Dataset with all events shape:", df_events.shape)

# Check generated event columns
event_cols_check = [
    # Price events
    "low_price",
    "high_price",
    "price_spike",
    "extreme_price",
    "rapid_price_change",
    "price_ramp_up",
    "price_ramp_down",
    "high_volatility",

    # Volume events
    "high_demand",
    "low_demand",
    "high_generation",
    "low_generation",
    "volume_imbalance",
    "strong_buy_pressure",
    "strong_sell_pressure",
    "buy_volume_spike",
    "sell_volume_spike",

    # Combined events
    "generation_surplus",
    "demand_pressure",
    "strong_demand_pressure",
    "strong_generation_pressure",
    "demand_driven_price_spike",
    "generation_driven_low_price",
    "scarcity_price_event",
    "oversupply_price_event",
    "buy_pressure_price_spike",
    "sell_pressure_low_price",
    "price_separation",
    "extreme_price_separation",
    "zone_price_outlier",
    "has_combined_event",
]

existing_event_cols = [col for col in event_cols_check if col in df_events.columns]

df_events[existing_event_cols].head()

Dataset with all events shape: (43805, 60)


,low_price,high_price,price_spike,extreme_price,rapid_price_change,price_ramp_up,price_ramp_down,high_volatility,high_demand,low_demand,...,demand_driven_price_spike,generation_driven_low_price,scarcity_price_event,oversupply_price_event,buy_pressure_price_spike,sell_pressure_low_price,price_separation,extreme_price_separation,zone_price_outlier,has_combined_event
datetime,,,,,,,,,,,,,,,,,,,,,
2020-01-01 00:00:00,False,True,True,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2020-01-01 01:00:00,False,True,True,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2020-01-01 02:00:00,False,True,True,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2020-01-01 03:00:00,False,True,True,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2020-01-01 04:00:00,False,True,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [23]:
# ============================================================
# 7. CYCLICAL TIME FEATURES
# ============================================================

df_events = df_events.copy()

df_events["hour"] = df_events.index.hour
df_events["dayofweek"] = df_events.index.dayofweek

df_events["hour_sin"] = np.sin(2 * np.pi * df_events["hour"] / 24)
df_events["hour_cos"] = np.cos(2 * np.pi * df_events["hour"] / 24)

df_events["dayofweek_sin"] = np.sin(2 * np.pi * df_events["dayofweek"] / 7)
df_events["dayofweek_cos"] = np.cos(2 * np.pi * df_events["dayofweek"] / 7)

df_events[
    [
        "hour",
        "hour_sin",
        "hour_cos",
        "dayofweek",
        "dayofweek_sin",
        "dayofweek_cos"
    ]
].head()

,hour,hour_sin,hour_cos,dayofweek,dayofweek_sin,dayofweek_cos
datetime,,,,,,
2020-01-01 00:00:00,0,0.000000,1.000000,2,0.974928,-0.222521
2020-01-01 01:00:00,1,0.258819,0.965926,2,0.974928,-0.222521
2020-01-01 02:00:00,2,0.500000,0.866025,2,0.974928,-0.222521
2020-01-01 03:00:00,3,0.707107,0.707107,2,0.974928,-0.222521
2020-01-01 04:00:00,4,0.866025,0.500000,2,0.974928,-0.222521


In [24]:
# ============================================================
# 7.1 ZONE ENCODING
# ============================================================

zone_dummies = pd.get_dummies(
    df_events["zone_id"],
    prefix="zone",
    drop_first=False
).astype(int)

df_events = pd.concat([df_events, zone_dummies], axis=1)

zone_features_lstm = list(zone_dummies.columns)

print("Zone features:", zone_features_lstm)
df_events[["zone_id"] + zone_features_lstm].head()

Zone features: ['zone_NO1', 'zone_NO2', 'zone_NO3', 'zone_NO4', 'zone_NO5']


,zone_id,zone_NO1,zone_NO2,zone_NO3,zone_NO4,zone_NO5
datetime,,,,,,
2020-01-01 00:00:00,NO1,1,0,0,0,0
2020-01-01 01:00:00,NO1,1,0,0,0,0
2020-01-01 02:00:00,NO1,1,0,0,0,0
2020-01-01 03:00:00,NO1,1,0,0,0,0
2020-01-01 04:00:00,NO1,1,0,0,0,0


In [ ]:
# ============================================================
# 8. DEFINE LSTM FEATURE SETS
# ============================================================

price_base_features_lstm = [
    "price_value",

    # Calendar features
    "year",
    "month",
    "day",
    "day_of_week",

    # Cyclical time features
    "hour_sin",
    "hour_cos",
    "dayofweek_sin",
    "dayofweek_cos",
] + zone_features_lstm


lag_features_lstm = [
    "price_value_lag_1",
    "price_value_lag_2",
    "price_value_lag_24",
    "price_value_lag_48",
    "price_value_lag_168",
]

volume_base_features_lstm = [
    "buy_volume_value",
    "sell_volume_value",
]

price_event_features_lstm = [
    "low_price",
    "high_price",
    "price_spike",
    "extreme_price",
    "rapid_price_change",
    "price_ramp_up",
    "price_ramp_down",
    "high_volatility",
]

volume_event_features_lstm = [
    "high_demand",
    "low_demand",
    "high_generation",
    "low_generation",
    "volume_imbalance",
    "strong_buy_pressure",
    "strong_sell_pressure",
    "buy_volume_delta",
    "sell_volume_delta",
    "abs_buy_volume_delta",
    "abs_sell_volume_delta",
    "buy_volume_spike",
    "sell_volume_spike",
]

combined_event_features_lstm = [
    "generation_surplus",
    "demand_pressure",
    "strong_demand_pressure",
    "strong_generation_pressure",
    "demand_driven_price_spike",
    "generation_driven_low_price",
    "scarcity_price_event",
    "oversupply_price_event",
    "buy_pressure_price_spike",
    "sell_pressure_low_price",
    "price_separation",
    "extreme_price_separation",
    "zone_price_outlier",
    "has_combined_event",
]

lstm_feature_sets = {
    "baseline_price_lags": (
        price_base_features_lstm
        + lag_features_lstm
    ),

    "baseline_price_lags_volume": (
        price_base_features_lstm
        + lag_features_lstm
        + volume_base_features_lstm
    ),

    "price_events": (
        price_base_features_lstm
        + lag_features_lstm
        + volume_base_features_lstm
        + price_event_features_lstm
    ),

    "price_volume_events": (
        price_base_features_lstm
        + lag_features_lstm
        + volume_base_features_lstm
        + price_event_features_lstm
        + volume_event_features_lstm
    ),


    "all_events": (
        price_base_features_lstm
        + lag_features_lstm
        + volume_base_features_lstm
        + price_event_features_lstm
        + volume_event_features_lstm
        + combined_event_features_lstm
    ),
}

target_col = "target"

# Check missing columns
for set_name, cols in lstm_feature_sets.items():
    missing_cols = [col for col in cols if col not in df_events.columns]
    print(set_name, "n_features:", len(cols), "missing:", missing_cols)

baseline_price_lags n_features: 19 missing: []
baseline_price_lags_volume n_features: 21 missing: []
price_events n_features: 29 missing: []
price_volume_events n_features: 42 missing: []
all_events n_features: 56 missing: []


In [32]:
# ============================================================
# 8.1 ADD BASELINE + COMBINED EVENTS FEATURE SET
# ============================================================

lstm_feature_sets["baseline_combined_events"] = (
    price_base_features_lstm
    + lag_features_lstm
    + combined_event_features_lstm
)

missing_cols = [
    col for col in lstm_feature_sets["baseline_combined_events"]
    if col not in df_events.columns
]

print("baseline_combined_events n_features:", len(lstm_feature_sets["baseline_combined_events"]))
print("missing:", missing_cols)

baseline_combined_events n_features: 33
missing: []


In [33]:
# ============================================================
# 9. TEMPORAL TRAIN / TEST SPLIT DATES
# ============================================================

train_end = "2020-11-30 23:00:00"
test_start = "2020-12-01 00:00:00"
test_end = "2020-12-08 00:00:00"

lookback = 24

print("Train end:", train_end)
print("Test period:", test_start, "to", test_end)
print("LSTM lookback:", lookback, "hours")

Train end: 2020-11-30 23:00:00
Test period: 2020-12-01 00:00:00 to 2020-12-08 00:00:00
LSTM lookback: 24 hours


In [34]:
# ============================================================
# 10. CREATE LSTM SEQUENCES FUNCTION BY ZONE
# ============================================================

def create_lstm_sequences_by_zone(df_lstm, feature_cols, target_col, lookback):
    X_seq = []
    y_seq = []
    seq_index = []
    seq_zone = []

    for zone_id, zone_df in df_lstm.groupby("zone_id"):
        zone_df = zone_df.sort_index()

        X_zone = zone_df[feature_cols]
        y_zone = zone_df[target_col]

        for i in range(lookback, len(zone_df)):
            X_seq.append(X_zone.iloc[i - lookback:i].values)
            y_seq.append(y_zone.iloc[i])
            seq_index.append(zone_df.index[i])
            seq_zone.append(zone_id)

    X_seq = np.array(X_seq)
    y_seq = np.array(y_seq)
    seq_index = pd.Index(seq_index)
    seq_zone = np.array(seq_zone)

    return X_seq, y_seq, seq_index, seq_zone

print("Grouped sequence function created successfully")

Grouped sequence function created successfully


In [28]:
# ============================================================
# 11. FUNCTION TO TRAIN AND EVALUATE LSTM BY FEATURE SET
# ============================================================

def evaluate_model(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred)
    }


def train_evaluate_lstm_feature_set(feature_set_name, feature_cols, lookback=24):

    print("=" * 70)
    print(f"Training LSTM feature set: {feature_set_name}")
    print("=" * 70)

    # Keep zone_id because sequences must be created separately per zone
    df_lstm = df_events[["zone_id"] + feature_cols + [target_col]].dropna().copy()

    X_raw = df_lstm[feature_cols]
    y_raw = df_lstm[target_col]

    # Temporal masks
    train_mask = df_lstm.index <= train_end
    test_mask = (df_lstm.index >= test_start) & (df_lstm.index <= test_end)

    X_train_raw = X_raw.loc[train_mask]
    y_train_raw = y_raw.loc[train_mask]

    print("Raw X:", X_raw.shape)
    print("Raw y:", y_raw.shape)
    print("Train:", X_train_raw.shape)
    print("Number of features:", len(feature_cols))

    # Scaling fitted only on training data
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    scaler_X.fit(X_train_raw)
    scaler_y.fit(y_train_raw.values.reshape(-1, 1))

    X_scaled = pd.DataFrame(
        scaler_X.transform(X_raw),
        index=X_raw.index,
        columns=X_raw.columns
    )

    y_scaled = pd.Series(
        scaler_y.transform(y_raw.values.reshape(-1, 1)).ravel(),
        index=y_raw.index,
        name=target_col
    )

    # Rebuild scaled dataframe including zone_id
    df_scaled = X_scaled.copy()
    df_scaled[target_col] = y_scaled
    df_scaled["zone_id"] = df_lstm["zone_id"].values

    # Create LSTM sequences by zone
    X_seq, y_seq, seq_index, seq_zone = create_lstm_sequences_by_zone(
        df_scaled,
        feature_cols,
        target_col,
        lookback
    )

    train_seq_mask = seq_index <= train_end
    test_seq_mask = (seq_index >= test_start) & (seq_index <= test_end)

    X_train = X_seq[train_seq_mask]
    y_train = y_seq[train_seq_mask]

    X_test = X_seq[test_seq_mask]
    y_test = y_seq[test_seq_mask]

    y_test_index = seq_index[test_seq_mask]
    y_test_zone = seq_zone[test_seq_mask]

    print("X_train_lstm:", X_train.shape)
    print("X_test_lstm:", X_test.shape)
    print("Test zones:", pd.Series(y_test_zone).value_counts().to_dict())

    # Define LSTM model
    n_timesteps = X_train.shape[1]
    n_features = X_train.shape[2]

    model = Sequential([
        LSTM(
            units=64,
            input_shape=(n_timesteps, n_features),
            return_sequences=False
        ),
        Dropout(0.2),
        Dense(32, activation="relu"),
        Dense(1)
    ])

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss="mse"
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True
    )

    history = model.fit(
        X_train,
        y_train,
        validation_split=0.2,
        epochs=100,
        batch_size=32,
        callbacks=[early_stop],
        shuffle=False,
        verbose=0
    )

    # Predict
    y_pred_scaled = model.predict(X_test, verbose=0).ravel()

    # Inverse scaling
    y_pred = scaler_y.inverse_transform(
        y_pred_scaled.reshape(-1, 1)
    ).ravel()

    y_test_real = scaler_y.inverse_transform(
        y_test.reshape(-1, 1)
    ).ravel()

    # Use MultiIndex to avoid duplicated timestamps across zones
    test_multi_index = pd.MultiIndex.from_arrays(
        [y_test_index, y_test_zone],
        names=["datetime", "zone_id"]
    )

    pred_series = pd.Series(
        y_pred,
        index=test_multi_index,
        name=f"LSTM_{feature_set_name}"
    )

    real_series = pd.Series(
        y_test_real,
        index=test_multi_index,
        name="Real"
    )

    metrics = evaluate_model(real_series, pred_series)

    result = {
        "feature_set": feature_set_name,
        "model": f"LSTM_{lookback}h_NO1_NO5",
        "n_features": len(feature_cols),
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "R2": metrics["R2"],
        "epochs_trained": len(history.history["loss"])
    }

    return model, history, real_series, pred_series, result

In [36]:
# ============================================================
# 12.2 TRAIN ONLY BASELINE + COMBINED EVENTS
# ============================================================

combined_model, combined_history, combined_real, combined_pred, combined_result = train_evaluate_lstm_feature_set(
    feature_set_name="baseline_combined_events",
    feature_cols=lstm_feature_sets["baseline_combined_events"],
    lookback=lookback
)

combined_result

Training LSTM feature set: baseline_combined_events
Raw X: (42960, 33)
Raw y: (42960,)
Train: (39360, 33)
Number of features: 33
X_train_lstm: (39240, 24, 33)
X_test_lstm: (845, 24, 33)
Test zones: {'NO1': 169, 'NO2': 169, 'NO3': 169, 'NO4': 169, 'NO5': 169}


c:\Users\HUGO\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


{'feature_set': 'baseline_combined_events',
 'model': 'LSTM_24h_NO1_NO5',
 'n_features': 33,
 'MAE': 1.096858921795907,
 'RMSE': np.float64(1.550433175426128),
 'R2': 0.8376084933853978,
 'epochs_trained': 16}

In [30]:
# ============================================================
# 12. TRAIN LSTM FOR ALL FEATURE SETS
# ============================================================

lstm_models_by_features = {}
lstm_histories_by_features = {}
lstm_real_by_features = {}
lstm_predictions_by_features = {}
lstm_results = []

for feature_set_name, feature_cols in lstm_feature_sets.items():

    model, history, real_series, pred_series, result = train_evaluate_lstm_feature_set(
        feature_set_name=feature_set_name,
        feature_cols=feature_cols,
        lookback=lookback
    )

    lstm_models_by_features[feature_set_name] = model
    lstm_histories_by_features[feature_set_name] = history
    lstm_real_by_features[feature_set_name] = real_series
    lstm_predictions_by_features[feature_set_name] = pred_series
    lstm_results.append(result)

    print("Finished:", feature_set_name)
    print(result)

Training LSTM feature set: baseline_price_lags
Raw X: (42960, 19)
Raw y: (42960,)
Train: (39360, 19)
Number of features: 19
X_train_lstm: (39240, 24, 19)
X_test_lstm: (845, 24, 19)
Test zones: {'NO1': 169, 'NO2': 169, 'NO3': 169, 'NO4': 169, 'NO5': 169}


c:\Users\HUGO\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Finished: baseline_price_lags
{'feature_set': 'baseline_price_lags', 'model': 'LSTM_24h_NO1_NO5', 'n_features': 19, 'MAE': 1.0435568036412348, 'RMSE': np.float64(1.5361331166580783), 'R2': 0.8405902393288222, 'epochs_trained': 20}
Training LSTM feature set: baseline_price_lags_volume
Raw X: (42960, 21)
Raw y: (42960,)
Train: (39360, 21)
Number of features: 21
X_train_lstm: (39240, 24, 21)
X_test_lstm: (845, 24, 21)
Test zones: {'NO1': 169, 'NO2': 169, 'NO3': 169, 'NO4': 169, 'NO5': 169}


c:\Users\HUGO\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Finished: baseline_price_lags_volume
{'feature_set': 'baseline_price_lags_volume', 'model': 'LSTM_24h_NO1_NO5', 'n_features': 21, 'MAE': 1.480942582079645, 'RMSE': np.float64(2.0126466923127824), 'R2': 0.7263519904463975, 'epochs_trained': 14}
Training LSTM feature set: price_events
Raw X: (42960, 29)
Raw y: (42960,)
Train: (39360, 29)
Number of features: 29
X_train_lstm: (39240, 24, 29)
X_test_lstm: (845, 24, 29)
Test zones: {'NO1': 169, 'NO2': 169, 'NO3': 169, 'NO4': 169, 'NO5': 169}


...

Finished: price_events
{'feature_set': 'price_events', 'model': 'LSTM_24h_NO1_NO5', 'n_features': 29, 'MAE': 1.464122408443654, 'RMSE': np.float64(1.9982619564646582), 'R2': 0.7302496317396857, 'epochs_trained': 11}
Training LSTM feature set: price_volume_events
Raw X: (42960, 42)
Raw y: (42960,)
Train: (39360, 42)
Number of features: 42
X_train_lstm: (39240, 24, 42)
X_test_lstm: (845, 24, 42)
Test zones: {'NO1': 169, 'NO2': 169, 'NO3': 169, 'NO4': 169, 'NO5': 169}


c:\Users\HUGO\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Finished: price_volume_events
{'feature_set': 'price_volume_events', 'model': 'LSTM_24h_NO1_NO5', 'n_features': 42, 'MAE': 1.904584817942783, 'RMSE': np.float64(2.388215265890023), 'R2': 0.614695442500663, 'epochs_trained': 11}
Training LSTM feature set: all_events
Raw X: (42960, 56)
Raw y: (42960,)
Train: (39360, 56)
Number of features: 56
X_train_lstm: (39240, 24, 56)
X_test_lstm: (845, 24, 56)
Test zones: {'NO1': 169, 'NO2': 169, 'NO3': 169, 'NO4': 169, 'NO5': 169}


c:\Users\HUGO\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Finished: all_events
{'feature_set': 'all_events', 'model': 'LSTM_24h_NO1_NO5', 'n_features': 56, 'MAE': 1.4881503859074157, 'RMSE': np.float64(1.9524421848702707), 'R2': 0.7424784540170895, 'epochs_trained': 12}


In [31]:
# ============================================================
# 13. LSTM FEATURE SET COMPARISON
# ============================================================

lstm_feature_comparison_df = pd.DataFrame(lstm_results)

lstm_feature_comparison_df = lstm_feature_comparison_df.sort_values(
    ["RMSE", "MAE"],
    ascending=True
)

lstm_feature_comparison_df

,feature_set,model,n_features,MAE,RMSE,R2,epochs_trained
0,baseline_price_lags,LSTM_24h_NO1_NO5,19,1.043557,1.536133,0.840590,20
4,all_events,LSTM_24h_NO1_NO5,56,1.488150,1.952442,0.742478,12
2,price_events,LSTM_24h_NO1_NO5,29,1.464122,1.998262,0.730250,11
1,baseline_price_lags_volume,LSTM_24h_NO1_NO5,21,1.480943,2.012647,0.726352,14
3,price_volume_events,LSTM_24h_NO1_NO5,42,1.904585,2.388215,0.614695,11


In [37]:
# ============================================================
# 13.1 COMPARE BASELINE, BASELINE + COMBINED EVENTS AND ALL EVENTS
# ============================================================

comparison_extra = pd.DataFrame(lstm_results + [combined_result])

comparison_extra = comparison_extra[
    comparison_extra["feature_set"].isin([
        "baseline_price_lags",
        "baseline_combined_events",
        "all_events"
    ])
].sort_values(["RMSE", "MAE"])

comparison_extra

,feature_set,model,n_features,MAE,RMSE,R2,epochs_trained
0,baseline_price_lags,LSTM_24h_NO1_NO5,19,1.043557,1.536133,0.840590,20
5,baseline_combined_events,LSTM_24h_NO1_NO5,33,1.096859,1.550433,0.837608,16
4,all_events,LSTM_24h_NO1_NO5,56,1.488150,1.952442,0.742478,12
